## 1. Load the Clustered Dataset

The clustering phase is now completed.
The dataset contains the patient clinical variables, the cluster assigned by K-Means, and the final `risk_category`.

In this notebook, we will use `risk_category` as the target variable for supervised classification.


In [19]:
import pandas as pd

df = pd.read_csv("../data/processed/diabetes_clustered.csv")

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Cluster,risk_category
0,6.0,148.0,72.0,35.0,264.8,33.6,0.627,50.0,2,Low Risk
1,1.0,85.0,66.0,29.0,65.2,26.6,0.351,31.0,0,Low Risk
2,8.0,183.0,64.0,30.6,198.2,23.3,0.672,32.0,2,Low Risk
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0,0,Low Risk
4,0.0,137.0,40.0,35.0,168.0,43.1,2.288,33.0,1,High Risk


In [20]:
df.shape

(768, 10)

In [21]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Cluster', 'risk_category'],
      dtype='str')

## 2. Define Features and Target

`X` contains the clinical variables used to make the prediction.

`y` contains the risk category that the model must predict.

We don't use `Cluster` as a feature because it was used to create `risk_category`.


In [22]:
clinical_columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age"
]

X = df[clinical_columns]
y = df["risk_category"]

In [23]:
X.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6.0,148.0,72.0,35.0,264.8,33.6,0.627,50.0
1,1.0,85.0,66.0,29.0,65.2,26.6,0.351,31.0
2,8.0,183.0,64.0,30.6,198.2,23.3,0.672,32.0
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0
4,0.0,137.0,40.0,35.0,168.0,43.1,2.288,33.0


In [24]:
y.head()

0     Low Risk
1     Low Risk
2     Low Risk
3     Low Risk
4    High Risk
Name: risk_category, dtype: str

## 3. Check Target Distribution

We check how many patients belong to each risk category.

This tells us whether the classes are balanced or imbalanced.


In [25]:
y.value_counts()

risk_category
Low Risk     568
High Risk    200
Name: count, dtype: int64

In [26]:
y.value_counts(normalize=True) * 100 # get percentage of each class

risk_category
Low Risk     73.958333
High Risk    26.041667
Name: proportion, dtype: float64

## 4. Train/Test Split

We split the dataset into training and testing data.

The model learns from the training data and is evaluated on the unseen test data.

`stratify=y` keeps a similar proportion of Low Risk and High Risk patients in both sets.


In [27]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y #preserves the existing ratio
)

In [28]:
X_train.shape , X_test.shape

((614, 8), (154, 8))

In [29]:
y_train.value_counts()

risk_category
Low Risk     454
High Risk    160
Name: count, dtype: int64

In [30]:
y_test.value_counts()

risk_category
Low Risk     114
High Risk     40
Name: count, dtype: int64

## 6. Handle Class Imbalance with Scaling and SMOTE

The training dataset contains fewer **High Risk** patients than **Low Risk** patients.

Before applying SMOTE, we standardize the numerical features because SMOTE uses distances to find similar minority-class samples.

The steps are:

1. Standardize the training and test features using `StandardScaler`.
2. Apply `SMOTE` only to the scaled training data.
3. Keep the test data unchanged because it must represent unseen real data.
4. Check that the two classes are balanced after SMOTE.

SMOTE creates synthetic samples for the minority class instead of simply duplicating existing samples.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # if we use fit_transform here, data leakage

In [32]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

In [33]:
y_train_smote.value_counts()

risk_category
Low Risk     454
High Risk    454
Name: count, dtype: int64

In [34]:
X_train_smote.shape, X_test_scaled.shape

((908, 8), (154, 8))